# Introduction to Analysing TrackMate Data

This notebook is a **gentle, generic introduction** to analysing the output of a
[TrackMate](https://imagej.net/plugins/trackmate/) tracking run.

TrackMate (a Fiji/ImageJ plugin) detects objects ("spots") in each frame of a
time-lapse and links them into "tracks". It can export two CSV tables:

- **`*_tracks.csv`**: one row per track, with summary measurements
  (duration, mean speed, displacement, confinement ratio, ...).
- **`*_spots.csv`**: one row per detection, with per-spot position, time,
  intensity and shape measurements.

By the end of this notebook you will know how to:

1. Load TrackMate's CSV exports with `pandas`.
2. Explore track-level statistics (duration, speed, displacement, confinement).
3. Work with per-spot data to reconstruct trajectories and compute
   instantaneous speeds and mean-squared displacement (MSD).

Everything here is deliberately **generic**: it only relies on the standard
columns TrackMate always writes, so it should work with any TrackMate export
regardless of your biological question.


In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(context="notebook", style="whitegrid")


## 1. The TrackMate CSV format

TrackMate writes CSVs with **four header rows** before the data:

1. feature names (e.g. `TRACK_ID`, `TRACK_MEAN_SPEED`),
2. human-readable names,
3. short names,
4. units.

`pandas` cannot handle this automatically, so we skip the last three header rows
with `skiprows=[1, 2, 3]` and use row 0 (the feature names) as column headers.


In [2]:
def read_trackmate_csv(path):
    """Read a TrackMate-exported CSV, keeping the feature names as headers."""
    return pd.read_csv(path, skiprows=[1, 2, 3])


## 2. Point the notebook at your data

Set `DATA_DIR` to the folder that contains your exported `*_tracks.csv` and
`*_spots.csv` files. The code below searches for them recursively, so a nested
folder structure (e.g. one subfolder per imaging position) works fine.


In [3]:
DATA_DIR = Path("./example_input")  # <- change this to your data

track_files = sorted(DATA_DIR.rglob("*_tracks.csv"))
spot_files = sorted(DATA_DIR.rglob("*_spots.csv"))

print(f"Found {len(track_files)} track file(s) and {len(spot_files)} spot file(s) in {DATA_DIR}")
if not track_files or not spot_files:
    print("No TrackMate CSV files found - check that DATA_DIR is correct.")


Found 0 track file(s) and 0 spot file(s) in TrackMate_Outputs
No TrackMate CSV files found - check that DATA_DIR is correct.


For simplicity we load the first of each file type. If you have many files
(e.g. one per position), you can loop over them and concatenate the results
with `pd.concat(...)`.


In [4]:
tracks = None
spots = None

if track_files and spot_files:
    tracks = read_trackmate_csv(track_files[0])
    spots = read_trackmate_csv(spot_files[0])
    print(f"Loaded tracks from {track_files[0].name}")
    print(f"Loaded spots  from {spot_files[0].name}")


In [5]:
if tracks is not None:
    tracks.head()


Key track columns (see the units in the CSV header):

| column | meaning |
|---|---|
| `TRACK_ID` | unique identifier for each track |
| `NUMBER_SPOTS` | how many detections make up the track |
| `TRACK_DURATION` | track length in time units |
| `TRACK_DISPLACEMENT` | straight-line distance between first and last spot |
| `TOTAL_DISTANCE_TRAVELED` | sum of step lengths along the track |
| `TRACK_MEAN_SPEED` | `TOTAL_DISTANCE_TRAVELED / TRACK_DURATION` |
| `CONFINEMENT_RATIO` | `TRACK_DISPLACEMENT / TOTAL_DISTANCE_TRAVELED` (0-1) |
| `LINEARITY_OF_FORWARD_PROGRESSION` | how straight the track is (1 = perfectly straight) |
| `TRACK_MEAN_QUALITY` | mean detection quality of the spots in the track |


In [6]:
if tracks is not None and not tracks.empty:
    print(f"{len(tracks)} tracks")
    tracks[["TRACK_DURATION", "TRACK_DISPLACEMENT", "TRACK_MEAN_SPEED", "CONFINEMENT_RATIO"]].describe()


## 3. Track-level statistics

A good first step is to look at the distribution of track durations; this
tells you how long objects were tracked for, and is a useful quality-control
measure (very short tracks are often tracking artefacts).


In [7]:
if tracks is not None and not tracks.empty:
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.histplot(tracks["TRACK_DURATION"], bins=30, ax=ax)
    ax.set_xlabel("Track duration")
    ax.set_ylabel("Number of tracks")
    plt.show()


Mean speed and confinement ratio are two common ways to characterise motion.
Confinement ratio ranges from 0 (back-and-forth motion) to 1 (perfectly
straight movement). Plotting it against mean speed separates objects by their
mode of motion.


In [8]:
if tracks is not None and not tracks.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    sns.histplot(tracks["TRACK_MEAN_SPEED"], bins=30, ax=axes[0])
    axes[0].set_xlabel("Mean speed")

    sns.histplot(tracks["CONFINEMENT_RATIO"], bins=30, ax=axes[1])
    axes[1].set_xlabel("Confinement ratio")

    plt.tight_layout()
    plt.show()


In [9]:
if tracks is not None and not tracks.empty:
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.scatterplot(
        data=tracks,
        x="TRACK_MEAN_SPEED",
        y="CONFINEMENT_RATIO",
        hue="TRACK_MEAN_QUALITY",
        ax=ax,
    )
    plt.show()


## 4. Reconstructing trajectories from spots

The `*_spots.csv` table contains every detection, with its `TRACK_ID`,
`POSITION_X` / `POSITION_Y` / `POSITION_Z`, `FRAME` and `POSITION_T` (time).
By plotting the spots of a few tracks we can visualise the paths the objects
took.


In [10]:
if spots is not None:
    spots.head()


Key spot columns:

| column | meaning |
|---|---|
| `TRACK_ID` | which track this spot belongs to |
| `POSITION_X`, `POSITION_Y`, `POSITION_Z` | spot coordinates |
| `FRAME` | frame number the spot was detected in |
| `POSITION_T` | time (see units) the spot was detected at |
| `QUALITY` | detection quality score |
| `RADIUS` | estimated spot radius |

Channel intensities (e.g. `MEAN_INTENSITY_CH1`) and shape features are also
exported when available; their names depend on your detector settings.


In [11]:
if spots is not None and not spots.empty:
    example_tracks = sorted(spots["TRACK_ID"].unique())[:5]

    fig, ax = plt.subplots(figsize=(8, 8))
    for track_id in example_tracks:
        track_spots = spots[spots["TRACK_ID"] == track_id].sort_values("FRAME")
        ax.plot(
            track_spots["POSITION_X"],
            track_spots["POSITION_Y"],
            marker="o",
            markersize=2,
            linewidth=1,
            label=f"Track {track_id}",
        )
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.legend()
    ax.set_aspect("equal")
    plt.show()


## 5. Instantaneous speed

TrackMate's `TRACK_MEAN_SPEED` is the *average* speed over a whole track
(total distance divided by duration). We can also compute the **instantaneous**
speed between consecutive frames from the spots data, which reveals how speed
changes over time.


In [12]:
spots_sorted = None

if spots is not None and not spots.empty:
    spots_sorted = spots.sort_values(["TRACK_ID", "FRAME"]).copy()
    by_track = spots_sorted.groupby("TRACK_ID")

    spots_sorted["DX"] = by_track["POSITION_X"].diff()
    spots_sorted["DY"] = by_track["POSITION_Y"].diff()
    spots_sorted["DT"] = by_track["POSITION_T"].diff()
    spots_sorted["INSTANT_SPEED"] = (
        np.sqrt(spots_sorted["DX"] ** 2 + spots_sorted["DY"] ** 2) / spots_sorted["DT"]
    )

    # The first spot of each track has no previous spot to compare with.
    spots_sorted["INSTANT_SPEED"] = spots_sorted["INSTANT_SPEED"].replace([np.inf, -np.inf], np.nan)

    print("Added DX, DY, DT and INSTANT_SPEED columns:")
    spots_sorted[["TRACK_ID", "FRAME", "POSITION_X", "POSITION_Y", "INSTANT_SPEED"]].head()


In [13]:
if spots_sorted is not None and not spots_sorted.empty:
    example_track = sorted(spots_sorted["TRACK_ID"].unique())[0]
    one_track = spots_sorted[spots_sorted["TRACK_ID"] == example_track]

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(one_track["FRAME"], one_track["INSTANT_SPEED"], marker="o", markersize=2)
    ax.set_xlabel("Frame")
    ax.set_ylabel("Instantaneous speed")
    ax.set_title(f"Track {example_track}")
    plt.show()


## 6. Mean squared displacement (MSD)

The **mean squared displacement** measures how far a particle has moved from its
starting point on average, as a function of time lag. For pure diffusion, MSD
grows linearly with lag; for directed (persistent) motion it grows faster.

We compute MSD per track, then plot a few examples. The lag axis is in frames;
multiply by the frame interval to convert to time units.


In [14]:
def track_msd(x, y):
    """Mean squared displacement vs. lag (in frames) for a single track."""
    x = np.asarray(x)
    y = np.asarray(y)
    n = len(x)
    msd = []
    for lag in range(1, n):
        dx = x[lag:] - x[:-lag]
        dy = y[lag:] - y[:-lag]
        msd.append(np.mean(dx ** 2 + dy ** 2))
    return msd


In [15]:
if spots_sorted is not None and not spots_sorted.empty:
    example_tracks = sorted(spots_sorted["TRACK_ID"].unique())[:4]

    fig, ax = plt.subplots(figsize=(8, 6))
    for track_id in example_tracks:
        track_spots = spots_sorted[spots_sorted["TRACK_ID"] == track_id].sort_values("FRAME")
        msd = track_msd(track_spots["POSITION_X"], track_spots["POSITION_Y"])
        ax.plot(range(1, len(msd) + 1), msd, label=f"Track {track_id}")

    ax.set_xlabel("Lag (frames)")
    ax.set_ylabel("Mean squared displacement")
    ax.legend()
    plt.show()


## 7. Morphological analysis

Beyond position and intensity, the `*_spots.csv` table often carries **shape**
features for each detection. Two of the most useful are `AREA` and
`CIRCULARITY`, but TrackMate can also export `PERIMETER`, `SOLIDITY` and
`ELLIPSE_ASPECTRATIO` (the exact set depends on your detector).

| column | meaning |
|---|---|
| `AREA` | projected area of the detection (see units) |
| `PERIMETER` | length of the detection outline |
| `CIRCULARITY` | `4 * pi * AREA / PERIMETER^2`, 0-1 (1 = perfect circle) |
| `SOLIDITY` | `AREA / convex-hull area` (1 = no concave regions) |
| `ELLIPSE_ASPECTRATIO` | long axis / short axis of the fitted ellipse |

`CIRCULARITY` and `SOLIDITY` are **size-independent** shape descriptors, so they
can be compared across objects of very different sizes.


In [16]:
if spots is not None and not spots.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    if "AREA" in spots.columns:
        sns.histplot(spots["AREA"], bins=30, ax=axes[0])
        axes[0].set_xlabel("Area")

    if "CIRCULARITY" in spots.columns:
        sns.histplot(spots["CIRCULARITY"], bins=30, ax=axes[1])
        axes[1].set_xlabel("Circularity")

    plt.tight_layout()
    plt.show()


In [17]:
if spots is not None and not spots.empty and {"AREA", "CIRCULARITY"} <= set(spots.columns):
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.scatterplot(
        data=spots,
        x="AREA",
        y="CIRCULARITY",
        hue="QUALITY",
        alpha=0.5,
        ax=ax,
    )
    plt.show()


## Summary & next steps

You have seen how to:

- load TrackMate's spots/tracks CSVs and understand their layout,
- summarise and visualise track-level statistics,
- reconstruct trajectories and instantaneous speeds from spots,
- compute mean squared displacement per track,
- explore per-spot shape features (area, circularity).

Things you might do next:

- **Compare conditions**: concatenate tracks from multiple files and add a
  condition/position column, then group by it.
- **Filter tracks**: e.g. keep only tracks longer than a minimum duration or
  above a quality threshold before drawing conclusions.
- **Plot intensity over time**: if your spots contain channel intensities
  (e.g. `MEAN_INTENSITY_CH1`), plot them per track to relate motion to
  fluorescent reporters.
- **Statistical testing**: use `scipy.stats` to test whether speed differs
  between groups.

The functions in this notebook are intentionally simple so they are easy to
adapt to your own experiment.
